# pSMAD_2024-08-21 — 04_image_preparation

**Feeds:** ED Fig 2j, 2k

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Image Preparation

## Notebook Scope

This notebook is a copied scaffold from the original workflow and has not yet been adapted for dataset 2. When adapted, it will prepare scene-level image panels from the dataset-2 cropped arrays.

## Setup

### Imports

In [ ]:
from pathlib import Path
import math
import re
import sys
from typing import Dict, Optional, Sequence

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import pandas as pd
from skimage import feature
from skimage import transform as sk_transform
from PIL import Image
from IPython.display import display

ROOT = Path('<analysis-root>/pSMAD')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.gradient_quantification_helpers import load_image_2d


### User Configuration

In [ ]:
ROOT = Path('<analysis-root>/pSMAD')
SCENE_IMAGE_DIR = ROOT / 'results/manual_bead_well_36locations/scene_images'
FIG1D_DIR = ROOT / 'results/figures/Fig1d_pSMAD_images'
FIG1D_DIR.mkdir(parents=True, exist_ok=True)
MONTAGE_DIAG_DIR = FIG1D_DIR / 'montage_diagnostics'
MONTAGE_DIAG_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR = ROOT / 'results/manual_bead_well_36locations/labels'
BEAD_WELL_TSV = ROOT / 'results/tables/manual_bead_well_positions_36locations.tsv'
ROI_SUMMARY_TSV = ROOT / 'results/tables/manual_roi_summary_36locations.tsv'
MANUAL_REVIEW_CSV = ROOT / 'results/tables/manual_gradient_manual_review.csv'

EXPORT_PNGS = True
SAVE_MONTAGE_PNG = True
RAW_MONTAGE_PNG_PATH = MONTAGE_DIAG_DIR / 'Fig1d_full_composite_montage_raw.png'
MASKED_MONTAGE_PNG_PATH = MONTAGE_DIAG_DIR / 'Fig1d_full_composite_montage_masked.png'
MASKED_LABEL_MONTAGE_PNG_PATH = MONTAGE_DIAG_DIR / 'Fig1d_full_composite_montage_masked_with_labels.png'
FULL_SCENE_MASKED_MONTAGE_PNG_PATH = FIG1D_DIR / 'Fig1d_full_composite_montage_masked_full_scene.png'
CENTERED_CANVAS_MASKED_MONTAGE_PNG_PATH = FIG1D_DIR / 'Fig1d_full_composite_montage_masked_centered_canvas.png'
MONTAGE_CENTER_ON_LABEL_ARRAY = True
MONTAGE_CROP_SIZE_PX = 1700
CENTERED_CANVAS_SIZE_PX = 2688
BEAD_WELL_ALPHA_FULL = 1.00
BEAD_WELL_ALPHA_PHASE = 0.82
BEAD_WELL_COLOR = np.array([1.0, 0.0, 0.0], dtype=np.float32)
BEAD_WELL_SEARCH_HALF_WINDOW_PX = 180
BEAD_WELL_RADII_PX = tuple(range(94, 115, 2))
BEAD_WELL_CANNY_SIGMA = 2.0
BEAD_WELL_TOTAL_NUM_PEAKS = 8
BEAD_WELL_MAX_CENTER_SHIFT_PX = 28.0

SCENE_PATTERN = re.compile(r'^scene(\d{2})_(bf|dapi|psmad|bead647)\.tif$')

# Global display scaling is estimated from a regular pixel subsample across all 36 scenes.
PERCENTILE_STEP = 16
BF_PERCENTILES = (1.0, 99.8)
DAPI_PERCENTILES = (1.0, 99.8)
PSMAD_PERCENTILES = (99.0, 99.8)

# Optional display gammas for aesthetics only.
BF_GAMMA = 1.0
DAPI_GAMMA = 1.0
PSMAD_GAMMA = 1.0
REPRESENTATIVE_SCENE_INDICES = [0, 9, 35]
DAPI_REVIEW_PERCENTILES = [(1.0, 99.5), (1.0, 99.8), (1.0, 99.95)]
PSMAD_REVIEW_PERCENTILES = [(97.0, 99.8), (98.0, 99.8), (99.0, 99.8)]
REVIEW_CROP_PADDING_PX = 140
BEAD_WELL_DEBUG_N_SCENES = 3
BEAD_WELL_DEBUG_SHOW_ALL = False
BEAD_WELL_DEBUG_BATCH_SIZE = 6
MANUAL_REVIEW_MASK_EXCEPTIONS = {'manual_s03_cyst_0001'}


### Helper Imports

In [ ]:
Image.MAX_IMAGE_PIXELS = None


def build_scene_table(scene_dir: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(scene_dir.glob('scene*_*.tif')):
        m = SCENE_PATTERN.match(path.name)
        if not m:
            continue
        scene_index = int(m.group(1))
        channel = m.group(2)
        rows.append({'scene_index': scene_index, 'channel': channel, 'path': path})
    if not rows:
        raise RuntimeError(f'No scene TIFFs found in {scene_dir}')
    df = pd.DataFrame(rows)
    pivot = (
        df.pivot(index='scene_index', columns='channel', values='path')
        .reset_index()
        .sort_values('scene_index')
        .reset_index(drop=True)
    )
    expected = ['bf', 'dapi', 'psmad']
    missing_cols = [c for c in expected if c not in pivot.columns]
    if missing_cols:
        raise RuntimeError(f'Missing scene channels: {missing_cols}')
    if pivot[expected].isna().any().any():
        raise RuntimeError('Some scene/channel paths are missing after pivot')
    pivot['scene_id'] = pivot['scene_index'].map(lambda i: f'scene{i:02d}')
    return pivot[['scene_index', 'scene_id', 'bf', 'dapi', 'psmad']]


def sampled_pixels(path: Path, step: int = 16) -> np.ndarray:
    img = load_image_2d(path)
    return np.asarray(img[::step, ::step], dtype=np.float32).ravel()


def pooled_sampled_pixels(paths, step: int = 16) -> np.ndarray:
    pooled = np.concatenate([sampled_pixels(Path(p), step=step) for p in paths])
    return pooled[np.isfinite(pooled)]


def normalize_for_display(img, lo, hi, gamma=1.0):
    img = np.asarray(img, dtype=np.float32)
    scaled = (img - float(lo)) / max(float(hi) - float(lo), 1e-6)
    scaled = np.clip(scaled, 0.0, 1.0)
    if gamma != 1.0:
        scaled = scaled ** float(gamma)
    return scaled


def grayscale_rgb(gray):
    gray = np.asarray(gray, dtype=np.float32)
    return np.dstack([gray, gray, gray])


def base_full_composite(dapi_img, psmad_img, scales, psmad_mask=None):
    dapi = normalize_for_display(dapi_img, *scales['dapi'], gamma=DAPI_GAMMA)
    psmad = normalize_for_display(psmad_img, *scales['psmad'], gamma=PSMAD_GAMMA)
    if psmad_mask is not None:
        psmad = np.where(np.asarray(psmad_mask, dtype=bool), psmad, 0.0)
    rgb = np.zeros(dapi.shape + (3,), dtype=np.float32)
    rgb += grayscale_rgb(dapi)
    rgb[..., 1] = np.clip(rgb[..., 1] + psmad, 0.0, 1.0)
    return np.clip(rgb, 0.0, 1.0)


def base_phase_panel(bf_img, scales):
    bf = normalize_for_display(bf_img, *scales['bf'], gamma=BF_GAMMA)
    return np.clip(grayscale_rgb(bf), 0.0, 1.0)


def single_channel_white(img, lo, hi, gamma=1.0):
    gray = normalize_for_display(img, lo, hi, gamma=gamma)
    return grayscale_rgb(gray)


def single_channel_green(img, lo, hi, gamma=1.0, mask=None):
    gray = normalize_for_display(img, lo, hi, gamma=gamma)
    if mask is not None:
        gray = np.where(np.asarray(mask, dtype=bool), gray, 0.0)
    rgb = np.zeros(gray.shape + (3,), dtype=np.float32)
    rgb[..., 1] = gray
    return rgb


def save_png(rgb, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.clip(np.round(np.asarray(rgb) * 255.0), 0, 255).astype(np.uint8)
    Image.fromarray(arr).save(path)


def load_roi_summary_table(roi_summary_tsv: Path) -> pd.DataFrame:
    df = pd.read_csv(roi_summary_tsv, sep='\t').copy()
    if 'scene_id' not in df.columns:
        df['scene_id'] = df['scene_index'].map(lambda i: f'scene{int(i):02d}')
    return df.sort_values(['scene_index', 'cyst_id']).reset_index(drop=True)


def load_manual_review_table(manual_review_csv: Path) -> pd.DataFrame:
    if not manual_review_csv.exists():
        return pd.DataFrame(columns=['cyst_id', 'review_action', 'include_in_downstream_analysis', 'review_reason'])
    return pd.read_csv(manual_review_csv).copy()


def label_path_for_scene(scene_id: str) -> Path:
    return LABEL_DIR / f"{scene_id}_cyst_labels.tif"


def label_image_for_scene(scene_id: str) -> np.ndarray:
    return np.asarray(Image.open(label_path_for_scene(scene_id)))


def label_mask_for_scene(scene_id: str) -> np.ndarray:
    return label_image_for_scene(scene_id) > 0


def _label_value_near_centroid(label_img: np.ndarray, centroid_x_px: float, centroid_y_px: float, search_radius_px: int = 6) -> int:
    h, w = label_img.shape[:2]
    x = int(round(float(centroid_x_px)))
    y = int(round(float(centroid_y_px)))
    x = max(0, min(x, w - 1))
    y = max(0, min(y, h - 1))
    direct = int(label_img[y, x])
    if direct > 0:
        return direct
    x0 = max(0, x - int(search_radius_px))
    x1 = min(w, x + int(search_radius_px) + 1)
    y0 = max(0, y - int(search_radius_px))
    y1 = min(h, y + int(search_radius_px) + 1)
    sub = label_img[y0:y1, x0:x1]
    vals = sub[sub > 0]
    if vals.size == 0:
        return 0
    uniq, counts = np.unique(vals.astype(int), return_counts=True)
    return int(uniq[np.argmax(counts)])


def build_psmad_mask_exclusion_records(manual_review_df: pd.DataFrame, roi_summary_df: pd.DataFrame, exception_cyst_ids=()):
    exception_cyst_ids = {str(c) for c in exception_cyst_ids}
    if len(manual_review_df) == 0:
        return [], {}
    excluded_df = manual_review_df.loc[~manual_review_df['include_in_downstream_analysis'].astype(bool)].copy()
    if len(excluded_df) == 0:
        return [], {}
    excluded_df = excluded_df.loc[~excluded_df['cyst_id'].astype(str).isin(exception_cyst_ids)].copy()
    records = []
    by_scene = {}
    for rr in excluded_df.itertuples(index=False):
        match = roi_summary_df.loc[roi_summary_df['cyst_id'].astype(str) == str(rr.cyst_id)]
        if len(match) == 0:
            continue
        row = match.iloc[0]
        scene_id = str(row['scene_id'])
        label_img = label_image_for_scene(scene_id)
        label_value = _label_value_near_centroid(label_img, float(row['centroid_x_px']), float(row['centroid_y_px']))
        records.append({
            'cyst_id': str(rr.cyst_id),
            'scene_id': scene_id,
            'scene_index': int(row['scene_index']),
            'label_value': int(label_value),
            'review_reason': str(getattr(rr, 'review_reason', '')),
        })
        if label_value > 0:
            by_scene.setdefault(scene_id, set()).add(int(label_value))
    return records, by_scene


def psmad_display_mask_for_scene(scene_id: str, excluded_label_map: dict[str, set[int]]) -> np.ndarray:
    label_img = label_image_for_scene(scene_id)
    mask = label_img > 0
    for label_value in sorted(excluded_label_map.get(scene_id, set())):
        mask &= label_img != int(label_value)
    return mask


def label_centroid_xy(label_path: Path) -> np.ndarray:
    label_img = np.asarray(Image.open(label_path))
    yy, xx = np.nonzero(label_img > 0)
    if len(xx) == 0:
        h, w = label_img.shape[:2]
        return np.array([w / 2.0, h / 2.0], dtype=np.float32)
    return np.array([float(xx.mean()), float(yy.mean())], dtype=np.float32)


def label_bbox_xyxy(label_path: Path):
    label_img = np.asarray(Image.open(label_path))
    yy, xx = np.nonzero(label_img > 0)
    h, w = label_img.shape[:2]
    if len(xx) == 0:
        return (0.0, 0.0, float(w), float(h))
    return (float(xx.min()), float(yy.min()), float(xx.max()) + 1.0, float(yy.max()) + 1.0)


def review_crop_bounds(image_shape, label_path: Path, bead_well_row: Optional[dict], crop_size: int, padding_px: int = 140):
    h, w = image_shape[:2]
    lx0, ly0, lx1, ly1 = label_bbox_xyxy(label_path)
    min_x = lx0 - float(padding_px)
    min_y = ly0 - float(padding_px)
    max_x = lx1 + float(padding_px)
    max_y = ly1 + float(padding_px)

    if bead_well_row and np.isfinite(bead_well_row.get('center_x_px', np.nan)) and np.isfinite(bead_well_row.get('center_y_px', np.nan)):
        r = float(bead_well_row.get('radius_px', bead_well_row.get('anchor_radius_px', np.mean(BEAD_WELL_RADII_PX))))
        bx = float(bead_well_row['center_x_px'])
        by = float(bead_well_row['center_y_px'])
        min_x = min(min_x, bx - r - float(padding_px))
        min_y = min(min_y, by - r - float(padding_px))
        max_x = max(max_x, bx + r + float(padding_px))
        max_y = max(max_y, by + r + float(padding_px))

    min_x = max(0.0, min_x)
    min_y = max(0.0, min_y)
    max_x = min(float(w), max_x)
    max_y = min(float(h), max_y)

    needed = max(max_x - min_x, max_y - min_y)
    crop_size = int(min(max(float(crop_size), float(needed)), float(min(h, w))))
    cx = 0.5 * (min_x + max_x)
    cy = 0.5 * (min_y + max_y)
    x0 = int(round(cx - crop_size / 2.0))
    y0 = int(round(cy - crop_size / 2.0))
    x0 = max(0, min(x0, w - crop_size))
    y0 = max(0, min(y0, h - crop_size))
    return x0, y0, x0 + crop_size, y0 + crop_size


def review_square_crop(img, label_path: Path, bead_well_row: Optional[dict], crop_size: int, padding_px: int = 140):
    arr = np.asarray(img)
    x0, y0, x1, y1 = review_crop_bounds(arr.shape, label_path, bead_well_row, crop_size, padding_px=padding_px)
    return arr[y0:y1, x0:x1], (x0, y0, x1, y1)

def center_image_on_label_canvas(img, label_path: Path, canvas_size: int):
    arr = np.asarray(img)
    h, w = arr.shape[:2]
    canvas_size = int(max(canvas_size, h, w))
    canvas = np.zeros((canvas_size, canvas_size, arr.shape[2]), dtype=arr.dtype)

    cx, cy = label_centroid_xy(label_path)
    target_x = 0.5 * canvas_size
    target_y = 0.5 * canvas_size
    offset_x = int(round(target_x - cx))
    offset_y = int(round(target_y - cy))

    dst_x0 = max(0, offset_x)
    dst_y0 = max(0, offset_y)
    src_x0 = max(0, -offset_x)
    src_y0 = max(0, -offset_y)
    copy_w = min(w - src_x0, canvas_size - dst_x0)
    copy_h = min(h - src_y0, canvas_size - dst_y0)
    if copy_w > 0 and copy_h > 0:
        canvas[dst_y0:dst_y0 + copy_h, dst_x0:dst_x0 + copy_w] = arr[src_y0:src_y0 + copy_h, src_x0:src_x0 + copy_w]
    return canvas

def load_manual_bead_well_anchor_table(bead_well_tsv: Path) -> pd.DataFrame:
    df = pd.read_csv(bead_well_tsv, sep='\t').copy()
    if 'scene_index' not in df.columns:
        raise RuntimeError(f'Missing scene_index in bead-well TSV: {bead_well_tsv}')
    df['scene_id'] = df['scene_index'].map(lambda i: f'scene{int(i):02d}')
    return df.sort_values('scene_index').reset_index(drop=True)


def available_bead_well_anchors(anchor_table_df: pd.DataFrame) -> pd.DataFrame:
    required = {'scene_index', 'scene_id', 'bead_well_x_px', 'bead_well_y_px'}
    missing = required.difference(anchor_table_df.columns)
    if missing:
        raise RuntimeError(f'Missing required bead-well anchor columns: {sorted(missing)}')

    out = anchor_table_df.copy()
    if 'bead_well_radius_px' not in out.columns:
        out['bead_well_radius_px'] = float(np.mean(BEAD_WELL_RADII_PX))
    if 'status' not in out.columns:
        out['status'] = 'annotated'

    mask = (
        np.isfinite(out['bead_well_x_px'])
        & np.isfinite(out['bead_well_y_px'])
        & out['status'].astype(str).ne('missing')
        & out['status'].astype(str).ne('pass')
    )

    cols = [
        'scene_index',
        'scene_id',
        'bead_well_x_px',
        'bead_well_y_px',
        'bead_well_radius_px',
        'status',
        'annotator',
        'updated_at_utc',
    ]
    cols = [c for c in cols if c in out.columns]
    return out.loc[mask, cols].reset_index(drop=True)


def run_local_well_circle_debug(
    image: np.ndarray,
    approx_x_px: float,
    approx_y_px: float,
    half_window_px: int = 180,
    radii_px: Sequence[int] = tuple(range(94, 115, 2)),
    canny_sigma: float = 2.0,
    total_num_peaks: int = 8,
    target_radius_px: Optional[float] = None,
) -> Dict[str, object]:
    arr = np.asarray(image, dtype=np.float32)
    H, W = arr.shape
    x = float(approx_x_px)
    y = float(approx_y_px)
    out = {
        'search_x_px': x,
        'search_y_px': y,
        'patch': None,
        'patch_n': None,
        'edges': None,
        'peak_df': pd.DataFrame(),
        'detected': None,
        'x0': None,
        'y0': None,
        'x1': None,
        'y1': None,
        'status': 'invalid_anchor',
    }
    if not np.isfinite(x) or not np.isfinite(y):
        return out
    if x < 0.0 or x >= float(W) or y < 0.0 or y >= float(H):
        return out

    xi = int(round(x))
    yi = int(round(y))
    half = int(max(24, half_window_px))
    x0 = max(0, xi - half)
    x1 = min(W, xi + half)
    y0 = max(0, yi - half)
    y1 = min(H, yi + half)
    patch = arr[y0:y1, x0:x1]
    out.update({'patch': patch, 'x0': x0, 'x1': x1, 'y0': y0, 'y1': y1, 'status': 'patch_ready'})
    finite = patch[np.isfinite(patch)]
    if finite.size < 64:
        out['status'] = 'insufficient_pixels'
        return out

    lo, hi = np.quantile(finite, [0.01, 0.99])
    patch_n = np.clip((patch - float(lo)) / (float(hi - lo) + 1e-6), 0.0, 1.0)
    edges = feature.canny(patch_n, sigma=float(canny_sigma))
    out.update({'patch_n': patch_n, 'edges': edges, 'status': 'edges_ready'})

    radii = np.asarray(list(radii_px), dtype=int)
    if radii.size == 0:
        out['status'] = 'no_radii'
        return out

    hough = sk_transform.hough_circle(edges, radii)
    accums, cx, cy, rad = sk_transform.hough_circle_peaks(
        hough, radii, total_num_peaks=int(max(1, total_num_peaks))
    )
    if len(cx) == 0:
        out['status'] = 'no_peaks'
        return out

    rows = []
    for peak_idx, (accum, cxi, cyi, rr) in enumerate(zip(accums, cx, cy, rad)):
        gx = float(x0 + float(cxi))
        gy = float(y0 + float(cyi))
        shift = float(np.hypot(gx - x, gy - y))
        radius_err = abs(float(rr) - float(target_radius_px)) if target_radius_px is not None and np.isfinite(target_radius_px) else np.nan
        rows.append({
            'peak_idx': int(peak_idx),
            'accum': float(accum),
            'local_x_px': float(cxi),
            'local_y_px': float(cyi),
            'x_px': gx,
            'y_px': gy,
            'radius_px': float(rr),
            'shift_px': shift,
            'radius_err_px': radius_err,
        })
    peak_df = pd.DataFrame(rows)
    if target_radius_px is not None and np.isfinite(target_radius_px):
        peak_df = peak_df.sort_values(['radius_err_px', 'shift_px', 'peak_idx']).reset_index(drop=True)
    else:
        peak_df = peak_df.sort_values(['shift_px', 'peak_idx']).reset_index(drop=True)
    best = dict(peak_df.iloc[0])
    out.update({
        'peak_df': peak_df,
        'detected': {
            'x_px': float(best['x_px']),
            'y_px': float(best['y_px']),
            'radius_px': float(best['radius_px']),
            'accum': float(best['accum']),
            'search_x_px': x,
            'search_y_px': y,
        },
        'status': 'ok',
    })
    return out


def _detect_local_well_circle_center(
    image: np.ndarray,
    approx_x_px: float,
    approx_y_px: float,
    half_window_px: int = 180,
    radii_px: Sequence[int] = tuple(range(94, 115, 2)),
    canny_sigma: float = 2.0,
    total_num_peaks: int = 8,
    target_radius_px: Optional[float] = None,
) -> Optional[Dict[str, float]]:
    debug = run_local_well_circle_debug(
        image=image,
        approx_x_px=approx_x_px,
        approx_y_px=approx_y_px,
        half_window_px=half_window_px,
        radii_px=radii_px,
        canny_sigma=canny_sigma,
        total_num_peaks=total_num_peaks,
        target_radius_px=target_radius_px,
    )
    return debug['detected']


def detect_bead_well_from_centroid_anchor(scene_id: str, bf_img: np.ndarray, anchor_row: Optional[dict]) -> dict:
    if not anchor_row:
        return {'scene_id': scene_id, 'status': 'missing_anchor'}

    anchor_x = float(anchor_row['bead_well_x_px'])
    anchor_y = float(anchor_row['bead_well_y_px'])
    anchor_radius = float(anchor_row.get('bead_well_radius_px', np.nan))
    if not np.isfinite(anchor_radius) or anchor_radius <= 0:
        anchor_radius = float(np.mean(BEAD_WELL_RADII_PX))

    detected = _detect_local_well_circle_center(
        image=bf_img,
        approx_x_px=anchor_x,
        approx_y_px=anchor_y,
        half_window_px=int(BEAD_WELL_SEARCH_HALF_WINDOW_PX),
        radii_px=BEAD_WELL_RADII_PX,
        canny_sigma=float(BEAD_WELL_CANNY_SIGMA),
        total_num_peaks=int(BEAD_WELL_TOTAL_NUM_PEAKS),
        target_radius_px=float(anchor_radius),
    )
    if detected is None:
        return {
            'scene_id': scene_id,
            'status': 'no_circle_detected',
            'source': 'manual_centroid_fallback',
            'anchor_x_px': anchor_x,
            'anchor_y_px': anchor_y,
            'anchor_radius_px': anchor_radius,
            'center_x_px': anchor_x,
            'center_y_px': anchor_y,
            'radius_px': anchor_radius,
            'center_shift_px': 0.0,
            'accum': np.nan,
        }

    shift_px = float(np.hypot(float(detected['x_px']) - anchor_x, float(detected['y_px']) - anchor_y))
    if shift_px > float(BEAD_WELL_MAX_CENTER_SHIFT_PX):
        return {
            'scene_id': scene_id,
            'status': 'circle_shift_too_large',
            'source': 'manual_centroid_fallback',
            'anchor_x_px': anchor_x,
            'anchor_y_px': anchor_y,
            'anchor_radius_px': anchor_radius,
            'center_x_px': anchor_x,
            'center_y_px': anchor_y,
            'radius_px': anchor_radius,
            'center_shift_px': shift_px,
            'accum': float(detected['accum']),
        }

    return {
        'scene_id': scene_id,
        'status': 'ok',
        'source': 'automated_local_hough',
        'anchor_x_px': anchor_x,
        'anchor_y_px': anchor_y,
        'anchor_radius_px': anchor_radius,
        'center_x_px': float(detected['x_px']),
        'center_y_px': float(detected['y_px']),
        'radius_px': float(detected['radius_px']),
        'center_shift_px': shift_px,
        'accum': float(detected['accum']),
    }


def circle_fill_mask(shape, center_x_px: float, center_y_px: float, radius_px: float):
    h, w = shape[:2]
    yy, xx = np.indices((h, w), dtype=np.float32)
    norm = ((xx + 0.5 - float(center_x_px)) / max(float(radius_px), 1.0)) ** 2 + ((yy + 0.5 - float(center_y_px)) / max(float(radius_px), 1.0)) ** 2
    return norm <= 1.0


def apply_bead_well_overlay(rgb, bead_well_row: Optional[dict], alpha=0.82, color=None):
    color = BEAD_WELL_COLOR if color is None else np.asarray(color, dtype=np.float32)
    out = np.asarray(rgb, dtype=np.float32).copy()
    if not bead_well_row or bead_well_row.get('status') == 'missing_anchor':
        return out
    fill_mask = circle_fill_mask(
        out.shape,
        center_x_px=float(bead_well_row['center_x_px']),
        center_y_px=float(bead_well_row['center_y_px']),
        radius_px=float(bead_well_row['radius_px']),
    )
    out[fill_mask] = (1.0 - float(alpha)) * out[fill_mask] + float(alpha) * color[None, :]
    return np.clip(out, 0.0, 1.0)


## Scene Inputs

### Scene Table

In [ ]:
scene_df = build_scene_table(SCENE_IMAGE_DIR)
print('Scenes found:', len(scene_df))
display(scene_df.head())

In [ ]:
roi_summary_df = load_roi_summary_table(ROI_SUMMARY_TSV)
manual_review_df = load_manual_review_table(MANUAL_REVIEW_CSV)
psmad_mask_exclusion_records, psmad_mask_exclusion_map = build_psmad_mask_exclusion_records(
    manual_review_df,
    roi_summary_df,
    exception_cyst_ids=MANUAL_REVIEW_MASK_EXCEPTIONS,
)

print('Masked pSMAD review exclusions applied by scene label:')
if len(psmad_mask_exclusion_records) == 0:
    print('  None')
else:
    display(pd.DataFrame(psmad_mask_exclusion_records))
    print('Exception cyst IDs kept in masked pSMAD outputs:', sorted(MANUAL_REVIEW_MASK_EXCEPTIONS))


## Display Scaling

### Global Channel Display Ranges

Each channel uses one pooled display range across all 36 scenes so the exported panels remain visually comparable.

In [ ]:
channel_samples = {
    'bf': pooled_sampled_pixels(scene_df['bf'], step=PERCENTILE_STEP),
    'dapi': pooled_sampled_pixels(scene_df['dapi'], step=PERCENTILE_STEP),
    'psmad': pooled_sampled_pixels(scene_df['psmad'], step=PERCENTILE_STEP),
}

channel_scales = {
    'bf': tuple(float(x) for x in np.percentile(channel_samples['bf'], BF_PERCENTILES)),
    'dapi': tuple(float(x) for x in np.percentile(channel_samples['dapi'], DAPI_PERCENTILES)),
    'psmad': tuple(float(x) for x in np.percentile(channel_samples['psmad'], PSMAD_PERCENTILES)),
}

scale_df = pd.DataFrame(
    [{'channel': key, 'lo': val[0], 'hi': val[1]} for key, val in channel_scales.items()]
)
anchor_table_df = load_manual_bead_well_anchor_table(BEAD_WELL_TSV)
anchor_df = available_bead_well_anchors(anchor_table_df)
anchor_map = {str(r['scene_id']): dict(r) for r in anchor_df.to_dict(orient='records')}

bead_well_detection_rows = []
for row in scene_df.itertuples(index=False):
    anchor_row = anchor_map.get(row.scene_id)
    if anchor_row is None:
        continue
    bf = load_image_2d(Path(row.bf))
    bead_well_detection_rows.append(
        detect_bead_well_from_centroid_anchor(row.scene_id, bf, anchor_row)
    )
bead_well_detection_df = pd.DataFrame(bead_well_detection_rows)
bead_well_detection_map = {str(r['scene_id']): dict(r) for r in bead_well_detection_rows}

print(f'Manual bead-well centroid anchors found: {len(anchor_df)} / {len(scene_df)} scenes')
if len(bead_well_detection_df):
    print(f'Automated local bead-well detections: {(bead_well_detection_df["status"] == "ok").sum()} ok / {len(bead_well_detection_df)} anchored scenes')
display(scale_df)
if len(anchor_df):
    display(anchor_df)
if len(bead_well_detection_df):
    display(bead_well_detection_df)


## Exports

### Per-Scene PNG Exports

For each scene, export four PNG panels:
- full composite: `DAPI` white + `pSMAD` green + refined bead-well overlay in red
- phase + refined bead-well overlay
- `DAPI` alone
- `pSMAD` alone


In [ ]:
export_rows = []

for row in scene_df.itertuples(index=False):
    scene_dir = FIG1D_DIR / row.scene_id
    bf = load_image_2d(Path(row.bf))
    dapi = load_image_2d(Path(row.dapi))
    psmad = load_image_2d(Path(row.psmad))
    label_mask = label_mask_for_scene(row.scene_id)
    psmad_display_mask = psmad_display_mask_for_scene(row.scene_id, psmad_mask_exclusion_map)
    bead_well_row = bead_well_detection_map.get(row.scene_id, None)

    full_rgb_raw = apply_bead_well_overlay(
        base_full_composite(dapi, psmad, channel_scales),
        bead_well_row,
        alpha=BEAD_WELL_ALPHA_FULL,
    )
    full_rgb_masked = apply_bead_well_overlay(
        base_full_composite(dapi, psmad, channel_scales, psmad_mask=psmad_display_mask),
        bead_well_row,
        alpha=BEAD_WELL_ALPHA_FULL,
    )
    phase_bead_rgb = apply_bead_well_overlay(
        base_phase_panel(bf, channel_scales),
        bead_well_row,
        alpha=BEAD_WELL_ALPHA_PHASE,
    )
    dapi_rgb = single_channel_white(dapi, *channel_scales['dapi'], gamma=DAPI_GAMMA)
    psmad_rgb_raw = single_channel_green(psmad, *channel_scales['psmad'], gamma=PSMAD_GAMMA)
    psmad_rgb_masked = single_channel_green(psmad, *channel_scales['psmad'], gamma=PSMAD_GAMMA, mask=psmad_display_mask)

    outputs = {
        'full_composite_raw': full_rgb_raw,
        'full_composite_masked': full_rgb_masked,
        'phase_bead': phase_bead_rgb,
        'dapi': dapi_rgb,
        'psmad_raw': psmad_rgb_raw,
        'psmad_masked': psmad_rgb_masked,
    }

    detection_status = bead_well_row.get('status', 'missing_anchor') if bead_well_row else 'missing_anchor'
    for stem, rgb in outputs.items():
        out_path = scene_dir / f'{row.scene_id}_{stem}.png'
        if EXPORT_PNGS:
            save_png(rgb, out_path)
        export_rows.append({
            'scene_id': row.scene_id,
            'panel': stem,
            'path': str(out_path),
            'bead_well_status': detection_status,
        })

export_df = pd.DataFrame(export_rows)
print('PNG exports prepared:', len(export_df))
print('Scenes with refined bead-well overlays:', export_df.loc[export_df['bead_well_status'] != 'missing_anchor', 'scene_id'].nunique())
display(export_df.head(18))


### Channel Intensity Histograms

These histograms use the same pooled sampled pixels used for display scaling.

- Solid red lines: current global linear display windows


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
axes = np.ravel(axes)
channel_order = ['bf', 'dapi', 'psmad']
channel_titles = {
    'bf': 'Brightfield',
    'dapi': 'DAPI',
    'psmad': 'pSMAD',
}

for ax, channel in zip(axes, channel_order):
    samples = channel_samples[channel]
    lo_view = float(np.percentile(samples, 0.1))
    hi_view = float(np.percentile(samples, 99.99))
    bins = np.linspace(lo_view, hi_view, 256)
    ax.hist(samples, bins=bins, color='0.35', alpha=0.95)
    ax.set_yscale('log')
    ax.set_title(channel_titles[channel])
    ax.set_xlabel('Pixel intensity')
    ax.set_ylabel('Count (log)')
    ax.set_xlim(lo_view, hi_view)

    lo, hi = channel_scales[channel]
    ax.axvline(lo, color='red', linewidth=1.5, label='Current global display window')
    ax.axvline(hi, color='red', linewidth=1.5)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

fig.suptitle('Channel intensity histograms', fontsize=16)
fig.text(
    0.5,
    0.01,
    'Solid red lines mark the current global display low/high cutoffs.',
    ha='center',
    va='bottom',
    fontsize=10,
)

plt.show()


### Representative Single-Channel Contrast Review

These review panels show three representative scenes in grayscale only.

Each column applies a different linear contrast window so cutoff choices can be judged directly on the raw channel images before looking at the merged montage.


In [ ]:
representative_scene_ids = [f"scene{i:02d}" for i in REPRESENTATIVE_SCENE_INDICES]
review_scene_df = scene_df.loc[scene_df['scene_index'].isin(REPRESENTATIVE_SCENE_INDICES)].copy()
review_scene_df = review_scene_df.sort_values('scene_index').reset_index(drop=True)

review_specs = [
    {
        'channel': 'dapi',
        'title': 'DAPI contrast review',
        'windows': DAPI_REVIEW_PERCENTILES,
        'mode': 'global',
    },
    {
        'channel': 'psmad',
        'title': 'pSMAD contrast review',
        'windows': PSMAD_REVIEW_PERCENTILES,
        'mode': 'global',
    },
]

for spec in review_specs:
    channel = spec['channel']
    windows = spec['windows']
    fig, axes = plt.subplots(
        len(review_scene_df),
        len(windows),
        figsize=(4.2 * len(windows), 4.2 * len(review_scene_df)),
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)

    global_windows = []
    samples = channel_samples[channel]
    for lo_q, hi_q in windows:
        lo, hi = np.percentile(samples, [lo_q, hi_q])
        global_windows.append((float(lo), float(hi)))

    for row_idx, row in enumerate(review_scene_df.itertuples(index=False)):
        img = load_image_2d(Path(getattr(row, channel)))
        for col_idx, (lo_q, hi_q) in enumerate(windows):
            ax = axes[row_idx, col_idx]
            lo, hi = global_windows[col_idx]
            gray = normalize_for_display(img, lo, hi, gamma=1.0)
            ax.imshow(gray, cmap='gray', vmin=0.0, vmax=1.0)
            if row_idx == 0:
                ax.set_title(f'q{lo_q:g}-q{hi_q:g}\n{lo:.0f}-{hi:.0f}', fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(row.scene_id, fontsize=10)
            ax.axis('off')

    fig.suptitle(spec['title'] + ' | global linear windows', fontsize=16)
    plt.show()


### Bead-Well Detection Debug

These panels show the local BF image-processing steps used to refine each manual `02c` bead-well centroid into a well circle. Only a few representative anchored scenes are shown here again, chosen from the low-, middle-, and high-shift detections. The all-scene view of the final `Hough circles on normalized patch` panel follows immediately below.

In [ ]:
ok_det_df = bead_well_detection_df.copy().sort_values(['status', 'center_shift_px', 'scene_id']).reset_index(drop=True)

if len(ok_det_df) == 0:
    print('No bead-well detections to debug.')
else:
    if BEAD_WELL_DEBUG_SHOW_ALL:
        debug_scene_ids = ok_det_df['scene_id'].tolist()
    else:
        ok_only = ok_det_df.loc[ok_det_df['status'] == 'ok'].copy().reset_index(drop=True)
        idxs = [0, len(ok_only) // 2, len(ok_only) - 1]
        idxs = sorted(set(int(i) for i in idxs if len(ok_only) > 0))[:BEAD_WELL_DEBUG_N_SCENES]
        debug_scene_ids = ok_only.iloc[idxs]['scene_id'].tolist()

    debug_scene_df = scene_df.loc[scene_df['scene_id'].isin(debug_scene_ids)].copy()
    debug_scene_df['scene_id'] = pd.Categorical(debug_scene_df['scene_id'], categories=debug_scene_ids, ordered=True)
    debug_scene_df = debug_scene_df.sort_values('scene_id').reset_index(drop=True)

    batch_size = int(BEAD_WELL_DEBUG_BATCH_SIZE) if BEAD_WELL_DEBUG_SHOW_ALL else len(debug_scene_df)
    batch_size = max(1, batch_size)

    print(f'Scenes shown in bead-well debug: {len(debug_scene_df)}')
    for batch_start in range(0, len(debug_scene_df), batch_size):
        batch_df = debug_scene_df.iloc[batch_start:batch_start + batch_size].copy().reset_index(drop=True)
        fig, axes = plt.subplots(len(batch_df), 5, figsize=(18, 4.2 * len(batch_df)), constrained_layout=True)
        axes = np.atleast_2d(axes)

        for row_idx, row in enumerate(batch_df.itertuples(index=False)):
            bf = load_image_2d(Path(row.bf)).astype(np.float32)
            det = bead_well_detection_map[row.scene_id]
            debug = run_local_well_circle_debug(
                image=bf,
                approx_x_px=float(det['anchor_x_px']),
                approx_y_px=float(det['anchor_y_px']),
                half_window_px=int(BEAD_WELL_SEARCH_HALF_WINDOW_PX),
                radii_px=BEAD_WELL_RADII_PX,
                canny_sigma=float(BEAD_WELL_CANNY_SIGMA),
                total_num_peaks=int(BEAD_WELL_TOTAL_NUM_PEAKS),
                target_radius_px=float(det['anchor_radius_px']),
            )
            label_path = label_path_for_scene(row.scene_id)
            bf_rgb = base_phase_panel(bf, channel_scales)
            crop_rgb, (x0c, y0c, x1c, y1c) = review_square_crop(
                bf_rgb, label_path, det, MONTAGE_CROP_SIZE_PX, padding_px=REVIEW_CROP_PADDING_PX
            )

            ax = axes[row_idx, 0]
            ax.imshow(crop_rgb)
            ax.scatter([float(det['anchor_x_px']) - x0c], [float(det['anchor_y_px']) - y0c], c='magenta', s=28, marker='x')
            ax.add_patch(Circle((float(det['center_x_px']) - x0c, float(det['center_y_px']) - y0c), radius=float(det['radius_px']), fill=False, edgecolor='red', linewidth=1.5))
            ax.set_title(f"{row.scene_id} review crop\nstatus={det['status']} | shift={det['center_shift_px']:.1f}px", fontsize=10)
            ax.axis('off')

            patch = debug.get('patch', None)
            patch_n = debug.get('patch_n', None)
            edges = debug.get('edges', None)
            peak_df = debug.get('peak_df', pd.DataFrame())

            ax = axes[row_idx, 1]
            if patch is not None:
                ax.imshow(patch, cmap='gray')
            ax.set_title('Local BF patch', fontsize=10)
            ax.axis('off')

            ax = axes[row_idx, 2]
            if patch_n is not None:
                ax.imshow(patch_n, cmap='gray', vmin=0.0, vmax=1.0)
            ax.set_title('Contrast-normalized patch', fontsize=10)
            ax.axis('off')

            ax = axes[row_idx, 3]
            if edges is not None:
                ax.imshow(edges, cmap='gray', vmin=0.0, vmax=1.0)
            ax.set_title('Canny edges', fontsize=10)
            ax.axis('off')

            ax = axes[row_idx, 4]
            if patch_n is not None:
                ax.imshow(patch_n, cmap='gray', vmin=0.0, vmax=1.0)
            if len(peak_df):
                for peak_row in peak_df.itertuples(index=False):
                    edge = 'red' if int(peak_row.peak_idx) == int(peak_df.iloc[0]['peak_idx']) else 'yellow'
                    lw = 1.8 if edge == 'red' else 0.8
                    alpha = 1.0 if edge == 'red' else 0.45
                    ax.add_patch(Circle((float(peak_row.local_x_px), float(peak_row.local_y_px)), radius=float(peak_row.radius_px), fill=False, edgecolor=edge, linewidth=lw, alpha=alpha))
            if debug.get('x0') is not None and debug.get('y0') is not None:
                ax.scatter([float(det['anchor_x_px']) - float(debug['x0'])], [float(det['anchor_y_px']) - float(debug['y0'])], c='magenta', s=20, marker='x')
            ax.set_title(f"Hough circles on normalized patch\nr={det['radius_px']:.1f}px | accum={det['accum']:.3f}", fontsize=10)
            ax.axis('off')

        fig.suptitle(f'Bead-well detection debug | scenes {batch_start + 1}-{batch_start + len(batch_df)} of {len(debug_scene_df)}', fontsize=15)
        plt.show()


### Hough Circle Montage

This montage shows the final `Hough circles on normalized patch` view for all scenes. Anchored scenes display the normalized local BF patch with all detected Hough circles, the selected well circle in red, and the `02c` anchor in magenta. Scenes without a `02c` bead-well anchor are shown as unavailable.

In [ ]:
montage_scene_df = scene_df.copy().sort_values('scene_index').reset_index(drop=True)
ncols = 6
nrows = int(np.ceil(len(montage_scene_df) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 3.0 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for ax, row in zip(axes, montage_scene_df.itertuples(index=False)):
    ax.set_axis_off()
    det = bead_well_detection_map.get(row.scene_id, None)
    if det is None or not np.isfinite(det.get('anchor_x_px', np.nan)) or str(det.get('status', '')) == 'missing_anchor':
        ax.text(0.5, 0.56, row.scene_id, ha='center', va='center', fontsize=10, transform=ax.transAxes)
        ax.text(0.5, 0.42, 'no 02c anchor', ha='center', va='center', fontsize=9, color='0.4', transform=ax.transAxes)
        continue

    bf = load_image_2d(Path(row.bf)).astype(np.float32)
    debug = run_local_well_circle_debug(
        image=bf,
        approx_x_px=float(det['anchor_x_px']),
        approx_y_px=float(det['anchor_y_px']),
        half_window_px=int(BEAD_WELL_SEARCH_HALF_WINDOW_PX),
        radii_px=BEAD_WELL_RADII_PX,
        canny_sigma=float(BEAD_WELL_CANNY_SIGMA),
        total_num_peaks=int(BEAD_WELL_TOTAL_NUM_PEAKS),
        target_radius_px=float(det['anchor_radius_px']),
    )
    patch_n = debug.get('patch_n', None)
    peak_df = debug.get('peak_df', pd.DataFrame())
    if patch_n is not None:
        ax.imshow(patch_n, cmap='gray', vmin=0.0, vmax=1.0)
    else:
        ax.text(0.5, 0.5, 'patch unavailable', ha='center', va='center', fontsize=9, color='0.4', transform=ax.transAxes)
        continue

    if len(peak_df):
        selected_peak_idx = int(peak_df.iloc[0]['peak_idx'])
        for peak_row in peak_df.itertuples(index=False):
            edge = 'red' if int(peak_row.peak_idx) == selected_peak_idx else 'yellow'
            lw = 1.8 if edge == 'red' else 0.8
            alpha = 1.0 if edge == 'red' else 0.45
            ax.add_patch(Circle((float(peak_row.local_x_px), float(peak_row.local_y_px)), radius=float(peak_row.radius_px), fill=False, edgecolor=edge, linewidth=lw, alpha=alpha))
    if debug.get('x0') is not None and debug.get('y0') is not None:
        ax.scatter([float(det['anchor_x_px']) - float(debug['x0'])], [float(det['anchor_y_px']) - float(debug['y0'])], c='magenta', s=16, marker='x')
    ax.set_title(row.scene_id + "\n" + f"r={det['radius_px']:.0f}px | shift={det['center_shift_px']:.1f}px", fontsize=9)

for ax in axes[len(montage_scene_df):]:
    ax.set_axis_off()

fig.suptitle('Bead-well Hough circles on normalized patch | all scenes', fontsize=16)
plt.show()

### Bead-Well Refinement Preview

These previews show the manual `02c` bead-well centroid anchor and the refined local BF circle detection used for the red bead-well overlay.


In [ ]:
if len(bead_well_detection_df) == 0:
    print('No manual bead-well centroid anchors found in the 02c table.')
else:
    preview_scene_df = scene_df.loc[scene_df['scene_id'].isin(bead_well_detection_df['scene_id'])].copy()
    preview_scene_df = preview_scene_df.sort_values('scene_index').reset_index(drop=True)

    fig, axes = plt.subplots(
        len(preview_scene_df),
        3,
        figsize=(12.5, 4.5 * len(preview_scene_df)),
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)

    for row_idx, row in enumerate(preview_scene_df.itertuples(index=False)):
        bf = load_image_2d(Path(row.bf))
        base = base_phase_panel(bf, channel_scales)
        det = bead_well_detection_map[row.scene_id]
        anchor_mask = circle_fill_mask(
            base.shape,
            center_x_px=float(det['anchor_x_px']),
            center_y_px=float(det['anchor_y_px']),
            radius_px=10.0,
        )
        anchor_rgb = base.copy()
        anchor_rgb[anchor_mask] = 0.20 * anchor_rgb[anchor_mask] + 0.80 * np.array([1.0, 0.25, 0.25], dtype=np.float32)[None, :]
        refined_rgb = apply_bead_well_overlay(base, det, alpha=BEAD_WELL_ALPHA_PHASE)

        label_path = label_path_for_scene(row.scene_id)
        base_crop, _ = review_square_crop(base, label_path, det, MONTAGE_CROP_SIZE_PX, padding_px=REVIEW_CROP_PADDING_PX)
        anchor_crop, _ = review_square_crop(anchor_rgb, label_path, det, MONTAGE_CROP_SIZE_PX, padding_px=REVIEW_CROP_PADDING_PX)
        refined_crop, _ = review_square_crop(refined_rgb, label_path, det, MONTAGE_CROP_SIZE_PX, padding_px=REVIEW_CROP_PADDING_PX)

        axes[row_idx, 0].imshow(base_crop)
        axes[row_idx, 0].set_title(f'{row.scene_id} phase', fontsize=11)
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(anchor_crop)
        axes[row_idx, 1].set_title(f'{row.scene_id} manual centroid anchor', fontsize=11)
        axes[row_idx, 1].axis('off')

        axes[row_idx, 2].imshow(refined_crop)
        axes[row_idx, 2].set_title(
            f"{row.scene_id} refined bead-well\nstatus={det['status']} | shift={det['center_shift_px']:.1f}px | r={det['radius_px']:.1f}px",
            fontsize=11,
        )
        axes[row_idx, 2].axis('off')

    plt.show()


### Composite Montages

These notebook-review montages show three scene-level composite variants, all cropped around the union of the cyst-array footprint and the refined bead-well when possible:

1. raw full composite
2. masked full composite, where only the `pSMAD` display is set to zero outside the precomputed DAPI-derived cyst labels
3. masked full composite with the cyst labels explicitly outlined


In [ ]:
n = len(scene_df)
cols = 6
rows = math.ceil(n / cols)

montage_specs = [
    ('full_composite_raw', 'Fig 1d raw full-composite montage', RAW_MONTAGE_PNG_PATH, False),
    ('full_composite_masked', 'Fig 1d masked full-composite montage', MASKED_MONTAGE_PNG_PATH, False),
    ('full_composite_masked', 'Fig 1d masked full-composite montage + cyst labels', MASKED_LABEL_MONTAGE_PNG_PATH, True),
]

for stem, title, out_path, draw_labels in montage_specs:
    fig, axes = plt.subplots(rows, cols, figsize=(16, 16), constrained_layout=True)
    axes = np.ravel(axes)

    for ax, row in zip(axes, scene_df.itertuples(index=False)):
        img = np.asarray(Image.open(FIG1D_DIR / row.scene_id / f'{row.scene_id}_{stem}.png'))
        det = bead_well_detection_map.get(row.scene_id, None)
        crop_bounds = None
        if MONTAGE_CENTER_ON_LABEL_ARRAY:
            img, crop_bounds = review_square_crop(img, label_path_for_scene(row.scene_id), det, MONTAGE_CROP_SIZE_PX, padding_px=REVIEW_CROP_PADDING_PX)
        ax.imshow(img)
        if draw_labels:
            label_img = np.asarray(Image.open(label_path_for_scene(row.scene_id)))
            if crop_bounds is not None:
                x0, y0, x1, y1 = crop_bounds
                label_img = label_img[y0:y1, x0:x1]
            if np.any(label_img > 0):
                ax.contour(label_img > 0, levels=[0.5], colors='deepskyblue', linewidths=0.6, alpha=0.9)
        ax.set_title(row.scene_id, fontsize=9, pad=3)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    fig.suptitle(title, fontsize=16)

    if SAVE_MONTAGE_PNG:
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        print('Saved montage PNG:', out_path)

    plt.show()


### Final Montage Options

These manuscript-style masked montages remove scene labels and cyst-label overlays, then compare two layout strategies that preserve identical magnification across all scenes:

1. full-scene tiles with no recentering
2. larger black tiles with per-scene translation so the DAPI-defined cyst array is centered without discarding image content

In [ ]:
montage_options = [
    ('Full-scene masked montage', FULL_SCENE_MASKED_MONTAGE_PNG_PATH, 'full_scene'),
    ('Centered-canvas masked montage', CENTERED_CANVAS_MASKED_MONTAGE_PNG_PATH, 'centered_canvas'),
]

for _title, _out_path, _mode in montage_options:
    fig, axes = plt.subplots(rows, cols, figsize=(13.2, 13.2), facecolor='white')
    axes = np.ravel(axes)

    for ax, row in zip(axes, scene_df.itertuples(index=False)):
        ax.set_facecolor('black')
        img = np.asarray(Image.open(FIG1D_DIR / row.scene_id / f'{row.scene_id}_full_composite_masked.png'))
        if _mode == 'centered_canvas':
            img = center_image_on_label_canvas(img, label_path_for_scene(row.scene_id), CENTERED_CANVAS_SIZE_PX)
        ax.imshow(img)
        ax.axis('off')

    for ax in axes[n:]:
        ax.set_facecolor('black')
        ax.axis('off')

    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.99, wspace=0.012, hspace=0.012)

    if SAVE_MONTAGE_PNG:
        fig.savefig(_out_path, dpi=300, bbox_inches='tight', pad_inches=0)
        print('Saved montage PNG:', _out_path)

    plt.show()